In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from sklearn.metrics import mean_squared_error

In [3]:
# Load the dataset
data = pd.read_csv('../../data/Data3.csv')

# Preprocess the data
# 1. Create a 'Site' column with the first 7 letters of the 'ERBS' column
data['Site'] = data['ERBS'].str[:7]

# 2. Create a 'Thana' column with the first 5 letters of the 'ERBS' column
data['Thana'] = data['ERBS'].str[:5]

# 3. Make 3 different sets of data grouped by 'EUTRANCELLFDD', 'Site', and 'Thana' with 'STARTTIME_DATE', then average the values

# Group by 'EUTRANCELLFDD' and 'STARTTIME_DATE'
data_grouped_eutrancellfdd = data.groupby(['EUTRANCELLFDD', 'STARTTIME_DATE']).mean().reset_index()

# Group by 'Site' and 'STARTTIME_DATE'
data_grouped_site = data.groupby(['Site', 'STARTTIME_DATE']).mean().reset_index()

# Group by 'Thana' and 'STARTTIME_DATE'
data_grouped_thana = data.groupby(['Thana', 'STARTTIME_DATE']).mean().reset_index()

C:\Users\HP\AppData\Local\Temp\ipykernel_155876\3725134306.py:14: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  data_grouped_eutrancellfdd = data.groupby(['EUTRANCELLFDD', 'STARTTIME_DATE']).mean().reset_index()
C:\Users\HP\AppData\Local\Temp\ipykernel_155876\3725134306.py:17: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  data_grouped_site = data.groupby(['Site', 'STARTTIME_DATE']).mean().reset_index()
C:\Users\HP\AppData\Local\Temp\ipykernel_155876\3725134306.py:20: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to Fal

In [4]:
# Function to create dataset for LSTM
def create_dataset(dataset, look_back=1):
    X, Y = [], []
    for i in range(len(dataset) - look_back - 1):
        a = dataset[i:(i + look_back), :]
        X.append(a)
        Y.append(dataset[i + look_back, :])
    return np.array(X), np.array(Y)

# Function to apply LSTM model and calculate MSE
def apply_lstm(data, feature_columns):
    # Select only the features of interest
    data = data[feature_columns]
    
    # Normalize the data
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)
    
    # Create LSTM dataset
    look_back = 1
    X, Y = create_dataset(scaled_data, look_back)
    
    # Split the data into training and testing sets
    train_size = int(len(X) * 0.8)
    test_size = len(X) - train_size
    X_train, X_test = X[:train_size], X[train_size:]
    Y_train, Y_test = Y[:train_size], Y[train_size:]
    
    # Reshape the input to be [samples, time steps, features]
    X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], X_train.shape[2]))
    X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], X_test.shape[2]))
    
    # Build LSTM model
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(LSTM(50, return_sequences=False))
    model.add(Dense(Y_train.shape[1]))
    
    model.compile(optimizer='adam', loss='mean_squared_error')
    
    # Train the model
    model.fit(X_train, Y_train, epochs=20, batch_size=1, verbose=2)
    
    # Make predictions
    train_predict = model.predict(X_train)
    test_predict = model.predict(X_test)
    
    # Inverse transform the predictions and actual values to original scale
    train_predict = scaler.inverse_transform(train_predict)
    Y_train = scaler.inverse_transform(Y_train)
    test_predict = scaler.inverse_transform(test_predict)
    Y_test = scaler.inverse_transform(Y_test)
    
    # Calculate MSE
    train_mse = mean_squared_error(Y_train, train_predict)
    test_mse = mean_squared_error(Y_test, test_predict)
    
    return train_mse, test_mse

# Apply LSTM to each of the grouped datasets

# Grouped by EUTRANCELLFDD
feature_columns = ['AVG_NO_USER', 'AVG_USR_THRPUT_DL', 'DL_TRAFFIC_MB']  # Example feature columns
train_mse_eutrancellfdd, test_mse_eutrancellfdd = apply_lstm(data_grouped_eutrancellfdd, feature_columns)

# Grouped by Site
train_mse_site, test_mse_site = apply_lstm(data_grouped_site, feature_columns)

# Grouped by Thana
train_mse_thana, test_mse_thana = apply_lstm(data_grouped_thana, feature_columns)

# Print MSE results
print(f'MSE for EUTRANCELLFDD Grouping - Train: {train_mse_eutrancellfdd}, Test: {test_mse_eutrancellfdd}')
print(f'MSE for Site Grouping - Train: {train_mse_site}, Test: {test_mse_site}')
print(f'MSE for Thana Grouping - Train: {train_mse_thana}, Test: {test_mse_thana}')

Epoch 1/20
288236/288236 - 582s - loss: 0.0011 - 582s/epoch - 2ms/step
Epoch 2/20
288236/288236 - 581s - loss: 0.0010 - 581s/epoch - 2ms/step
Epoch 3/20
288236/288236 - 581s - loss: 0.0010 - 581s/epoch - 2ms/step
Epoch 4/20
288236/288236 - 582s - loss: 9.9412e-04 - 582s/epoch - 2ms/step
Epoch 5/20
288236/288236 - 584s - loss: 9.9029e-04 - 584s/epoch - 2ms/step
Epoch 6/20
288236/288236 - 583s - loss: 9.8687e-04 - 583s/epoch - 2ms/step
Epoch 7/20
288236/288236 - 583s - loss: 9.8439e-04 - 583s/epoch - 2ms/step
Epoch 8/20
288236/288236 - 593s - loss: 9.8173e-04 - 593s/epoch - 2ms/step
Epoch 9/20
288236/288236 - 583s - loss: 9.8028e-04 - 583s/epoch - 2ms/step
Epoch 10/20
288236/288236 - 559s - loss: 9.7902e-04 - 559s/epoch - 2ms/step
Epoch 11/20
288236/288236 - 562s - loss: 9.7815e-04 - 562s/epoch - 2ms/step
Epoch 12/20
288236/288236 - 559s - loss: 9.7629e-04 - 559s/epoch - 2ms/step
Epoch 13/20
288236/288236 - 655s - loss: 9.7502e-04 - 655s/epoch - 2ms/step
Epoch 14/20
288236/288236 - 657s 

In [6]:
import matplotlib.pyplot as plt

# Function to plot actual vs. predicted values
def plot_actual_vs_predicted(y_test, test_predict, title):
    plt.figure(figsize=(10, 6))
    plt.plot(y_test[:, 0], label='Actual', color='blue')  # Plot the first feature for simplicity
    plt.plot(test_predict[:, 0], label='Predicted', color='red')
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.legend()
    plt.show()

# Assume you already have Y_test and test_predict from your LSTM model

# Example for EUTRANCELLFDD Grouping
# train_mse_eutrancellfdd, test_mse_eutrancellfdd, Y_test_eutrancellfdd, test_predict_eutrancellfdd = apply_lstm(data_grouped_eutrancellfdd, feature_columns)
plot_actual_vs_predicted(Y_test_eutrancellfdd, test_predict_eutrancellfdd, 'Actual vs Predicted - EUTRANCELLFDD Grouping')

# Example for Site Grouping
# train_mse_site, test_mse_site, Y_test_site, test_predict_site = apply_lstm(data_grouped_site, feature_columns)
plot_actual_vs_predicted(Y_test_site, test_predict_site, 'Actual vs Predicted - Site Grouping')

# Example for Thana Grouping
# train_mse_thana, test_mse_thana, Y_test_thana, test_predict_thana = apply_lstm(data_grouped_thana, feature_columns)
plot_actual_vs_predicted(Y_test_thana, test_predict_thana, 'Actual vs Predicted - Thana Grouping')

NameError: name 'Y_test_eutrancellfdd' is not defined